In [ ]:
import torch

print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
else:
    print("No se detectó una GPU.")

GPU disponible: True
GPU: Tesla T4
CUDA: 12.8


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo utilizado:", device)

Dispositivo utilizado: cuda


In [ ]:
!pip install -q transformers datasets evaluate scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd

from datasets import load_dataset
from transformers import pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Bibliotecas cargadas correctamente.")

Bibliotecas cargadas correctamente.


In [ ]:
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import whoami

usuario = whoami()

print("Usuario autenticado:", usuario["name"])

Usuario autenticado: FernandoTec01


In [ ]:
from datasets import load_dataset

# Cargar el dataset de Amazon Polarity
dataset = load_dataset("mteb/amazon_polarity")

print(dataset)

README.md:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  255MB            

data/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  254MB            

data/train-00001-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  115MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3599994 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'text', 'label_text'],
        num_rows: 3599994
    })
    test: Dataset({
        features: ['label', 'text', 'label_text'],
        num_rows: 400000
    })
})


In [ ]:
# Consultar las divisiones del dataset
print("Divisiones:", dataset.keys())

# Consultar las columnas
print("Columnas:", dataset["test"].column_names)

# Mostrar un ejemplo
print("\nEjemplo de reseña:")
print(dataset["test"][0])

Divisiones: dict_keys(['train', 'test'])
Columnas: ['label', 'text', 'label_text']

Ejemplo de reseña:
{'label': 1, 'text': 'Great CD\n\nMy lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I\'m in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kill. One of life\'s hidden gems. This is a desert isle CD in my book. Why she never made it big is just beyond me. Everytime I play this, no matter black, white, young, old, male, female EVERYBODY says one thing "Who was that singing ?"', 'label_text': 'positive'}


In [ ]:
# Consultar las etiquetas disponibles
print(dataset["test"].features["label"])

# Mostrar las etiquetas de un ejemplo
print("Texto:", dataset["test"][0]["text"][:300])
print("Etiqueta:", dataset["test"][0]["label"])

Value('int64')
Texto: Great CD

My lovely Pat has one of the GREAT voices of her generation. I have listened to this CD for YEARS and I still LOVE IT. When I'm in a good mood it makes me feel better. A bad mood just evaporates like sugar in the rain. This CD just oozes LIFE. Vocals are jusat STUUNNING and lyrics just kil
Etiqueta: 1


In [ ]:
# Número de reseñas que evaluaremos inicialmente
N_MUESTRA = 1000

# Semilla para reproducir la selección
SEMILLA = 42

# Seleccionar una muestra aleatoria del conjunto de prueba
muestra = dataset["test"].shuffle(
    seed=SEMILLA
).select(range(N_MUESTRA))

print("Número de reseñas:", len(muestra))
print("Primera reseña:", muestra[0]["text"][:200])
print("Etiqueta original:", muestra[0]["label"])

Número de reseñas: 1000
Primera reseña: Damaged Product

The product works fine. I ordered the more exprensive one after I read reviews from others on Amazon. My husband likes the presser. It does a good job pressing his pants. However, it 
Etiqueta original: 1


In [ ]:
# Identificadores de los modelos en Hugging Face Hub

modelos = {
    "TinyBERT IMDB":
        "Harsha901/tinybert-imdb-sentiment-analysis-model",

    "DistilBERT SST-2":
        "distilbert/distilbert-base-uncased-finetuned-sst-2-english",

    "DistilBERT Product Reviews":
        "Floressek/sentiment_classification_from_distillbert"
}

print("Modelos seleccionados:")
for nombre, identificador in modelos.items():
    print(nombre, "->", identificador)

Modelos seleccionados:
TinyBERT IMDB -> Harsha901/tinybert-imdb-sentiment-analysis-model
DistilBERT SST-2 -> distilbert/distilbert-base-uncased-finetuned-sst-2-english
DistilBERT Product Reviews -> Floressek/sentiment_classification_from_distillbert


In [ ]:
from transformers import pipeline

# Diccionario donde guardaremos los clasificadores
clasificadores = {}

# Seleccionar GPU si está disponible
dispositivo = 0 if torch.cuda.is_available() else -1

# Cargar los tres modelos
for nombre, identificador in modelos.items():

    print(f"Cargando: {nombre}")

    clasificador = pipeline(
        "text-classification",
        model=identificador,
        tokenizer=identificador,
        device=dispositivo
    )

    clasificadores[nombre] = clasificador

    print(f"{nombre} cargado correctamente.\n")

print("Los tres modelos están listos.")

Cargando: TinyBERT IMDB


config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 57.4MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/73 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

DistilBERT Product Reviews cargado correctamente.

Los tres modelos están listos.


In [ ]:
texto_prueba = (
    "The product arrived quickly and works perfectly. "
    "I am very satisfied with my purchase."
)

for nombre, clasificador in clasificadores.items():

    resultado = clasificador(texto_prueba)

    print(f"\nModelo: {nombre}")
    print("Resultado:", resultado)


Modelo: TinyBERT IMDB
Resultado: [{'label': 'positive', 'score': 0.987744152545929}]

Modelo: DistilBERT SST-2
Resultado: [{'label': 'POSITIVE', 'score': 0.9998667240142822}]

Modelo: DistilBERT Product Reviews
Resultado: [{'label': 'LABEL_1', 'score': 0.9996104836463928}]


In [ ]:
# Convertir las reseñas a una lista de textos
textos = list(muestra["text"])

print("Tipo de datos:", type(textos))
print("Número de textos:", len(textos))
print("Tipo del primer texto:", type(textos[0]))
print("Primer texto:", textos[0][:200])

Tipo de datos: <class 'list'>
Número de textos: 1000
Tipo del primer texto: <class 'str'>
Primer texto: Damaged Product

The product works fine. I ordered the more exprensive one after I read reviews from others on Amazon. My husband likes the presser. It does a good job pressing his pants. However, it 


In [ ]:
# Pipeline de inferencia sobre las 1,000 reseñas

predicciones = {}

for nombre, clasificador in clasificadores.items():

    print(f"\nEvaluando: {nombre}")

    resultados = clasificador(
        textos,
        batch_size=16,
        truncation=True
    )

    predicciones[nombre] = resultados

    print("Número de predicciones:", len(resultados))
    print("Primer resultado:", resultados[0])


Evaluando: TinyBERT IMDB
Número de predicciones: 1000
Primer resultado: {'label': 'negative', 'score': 0.9569722414016724}

Evaluando: DistilBERT SST-2
Número de predicciones: 1000
Primer resultado: {'label': 'NEGATIVE', 'score': 0.9961140155792236}

Evaluando: DistilBERT Product Reviews
Número de predicciones: 1000
Primer resultado: {'label': 'LABEL_0', 'score': 0.535512387752533}


In [ ]:
# Convertir las predicciones de los tres modelos a valores 0 y 1

predicciones_numericas = {}

for nombre, resultados in predicciones.items():

    predicciones_modelo = []

    for resultado in resultados:

        etiqueta = resultado["label"].lower()

        if etiqueta in ["negative", "label_0"]:
            predicciones_modelo.append(0)

        elif etiqueta in ["positive", "label_1"]:
            predicciones_modelo.append(1)

        else:
            raise ValueError(f"Etiqueta desconocida: {resultado['label']}")

    predicciones_numericas[nombre] = predicciones_modelo

    print(f"{nombre}: {len(predicciones_modelo)} predicciones convertidas")

TinyBERT IMDB: 1000 predicciones convertidas
DistilBERT SST-2: 1000 predicciones convertidas
DistilBERT Product Reviews: 1000 predicciones convertidas


In [ ]:
# Obtener las etiquetas reales de las 1,000 reseñas

etiquetas_reales = muestra["label"]

print("Número de etiquetas reales:", len(etiquetas_reales))
print("Primeras 10 etiquetas:", etiquetas_reales[:10])

# Cálculo de métricas de evaluación

metricas = {}

for nombre, predicciones_modelo in predicciones_numericas.items():

    accuracy = accuracy_score(etiquetas_reales, predicciones_modelo)
    precision = precision_score(
        etiquetas_reales,
        predicciones_modelo,
        pos_label=1
    )
    recall = recall_score(
        etiquetas_reales,
        predicciones_modelo,
        pos_label=1
    )
    f1 = f1_score(
        etiquetas_reales,
        predicciones_modelo,
        pos_label=1
    )

    metricas[nombre] = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1
    }

    print(f"\nModelo: {nombre}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

Número de etiquetas reales: 1000
Primeras 10 etiquetas: [1, 0, 0, 0, 0, 1, 1, 1, 1, 1]

Modelo: TinyBERT IMDB
Accuracy : 0.9060
Precision: 0.9132
Recall   : 0.9025
F1-score : 0.9078

Modelo: DistilBERT SST-2
Accuracy : 0.9010
Precision: 0.9295
Recall   : 0.8733
F1-score : 0.9005

Modelo: DistilBERT Product Reviews
Accuracy : 0.9090
Precision: 0.8702
Recall   : 0.9669
F1-score : 0.9160


In [ ]:
import time

# Diccionario para almacenar los tiempos de inferencia
latencias = {}

print("Preparación para la medición de latencia completada.")

Preparación para la medición de latencia completada.


In [ ]:
# Medición de latencia de cada modelo

for nombre, clasificador in clasificadores.items():

    print(f"\nMidiendo latencia: {nombre}")

    # Sincronizar GPU antes de iniciar la medición
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    inicio = time.perf_counter()

    # Inferencia sobre las mismas 1,000 reseñas
    resultados = clasificador(
        textos,
        batch_size=16,
        truncation=True
    )

    # Sincronizar GPU antes de detener el cronómetro
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    fin = time.perf_counter()

    tiempo_total = fin - inicio
    latencias[nombre] = tiempo_total

    print(f"Tiempo total: {tiempo_total:.4f} segundos")


Midiendo latencia: TinyBERT IMDB
Tiempo total: 2.2369 segundos

Midiendo latencia: DistilBERT SST-2
Tiempo total: 6.5565 segundos

Midiendo latencia: DistilBERT Product Reviews
Tiempo total: 6.6010 segundos


In [ ]:
# Calcular latencia promedio por reseña

latencia_promedio = {}

for nombre, tiempo_total in latencias.items():

    tiempo_por_reseña = (tiempo_total / len(textos)) * 1000
    latencia_promedio[nombre] = tiempo_por_reseña

    print(
        f"{nombre}: "
        f"{tiempo_por_reseña:.4f} ms por reseña"
    )

TinyBERT IMDB: 2.2369 ms por reseña
DistilBERT SST-2: 6.5565 ms por reseña
DistilBERT Product Reviews: 6.6010 ms por reseña


In [ ]:
# Crear tabla resumen de resultados

resumen = []

for nombre in modelos.keys():
    resumen.append({
        "Modelo": nombre,
        "Accuracy": metricas[nombre]["Accuracy"],
        "Precision": metricas[nombre]["Precision"],
        "Recall": metricas[nombre]["Recall"],
        "F1-score": metricas[nombre]["F1-score"],
        "Latencia total (s)": latencias[nombre],
        "Latencia por reseña (ms)": latencia_promedio[nombre]
    })

tabla_resumen = pd.DataFrame(resumen)

# Mostrar valores con 4 decimales
tabla_resumen = tabla_resumen.round(4)

display(tabla_resumen)

,Modelo,Accuracy,Precision,Recall,F1-score,Latencia total (s),Latencia por reseña (ms)
0,TinyBERT IMDB,0.906,0.9132,0.9025,0.9078,2.2369,2.2369
1,DistilBERT SST-2,0.901,0.9295,0.8733,0.9005,6.5565,6.5565
2,DistilBERT Product Reviews,0.909,0.8702,0.9669,0.9160,6.6010,6.6010
